<a href="https://colab.research.google.com/github/firdoushkhilji/firdoushkhilji-7006SCN_FK_16943920/blob/task5/Task5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Task 5 — Model Evaluation & Stability

!pip install pyspark shap -q

from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("NHS_Task5_ModelEvaluation") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("SparkSession started!")

SparkSession started!


In [2]:
#import dataset
from google.colab import drive
drive.mount('/content/drive')

train_df = spark.read.parquet('/content/drive/MyDrive/TASK_DATASET/train_data.parquet')
test_df = spark.read.parquet('/content/drive/MyDrive/TASK_DATASET/test_data.parquet')

train_sample = train_df.sample(0.03, seed=42)
test_sample = test_df.sample(0.03, seed=42)

print(f"Training sample: {train_sample.count():,}")
print(f"Testing sample: {test_sample.count():,}")

Mounted at /content/drive
Training sample: 44,449
Testing sample: 11,190


In [3]:
#retraining all 4 models
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, MulticlassClassificationEvaluator

#1. linear regression
lr_model = LinearRegression(featuresCol='scaled_features', labelCol='ACTUAL_COST', regParam=0.01, elasticNetParam=0.0).fit(train_sample)
#2. random forest regressor
rf_model = RandomForestRegressor(featuresCol='scaled_features', labelCol='ACTUAL_COST', numTrees=20, maxDepth=5, seed=42).fit(train_sample)
#3. logistic regression
log_model = LogisticRegression(featuresCol='scaled_features', labelCol='HIGH_COST', regParam=0.01, elasticNetParam=0.0, maxIter=10).fit(train_sample)
#4. decision tree
dt_model = DecisionTreeClassifier(featuresCol='scaled_features', labelCol='HIGH_COST', maxDepth=5, seed=42).fit(train_sample)

print("All 4 models retrained using best known hyperparameters!")

All 4 models retrained using best known hyperparameters!


In [4]:
#metrics table for all models using test sample
#1. linear regression
lr_pred = lr_model.transform(test_sample)
#2. random forest regressor
rf_pred = rf_model.transform(test_sample)
#3. logistic regression
log_pred = log_model.transform(test_sample)
#4. decision tree
dt_pred = dt_model.transform(test_sample)

#evaluation
reg_eval_rmse = RegressionEvaluator(labelCol='ACTUAL_COST', metricName='rmse')
reg_eval_r2 = RegressionEvaluator(labelCol='ACTUAL_COST', metricName='r2')
reg_eval_mae = RegressionEvaluator(labelCol='ACTUAL_COST', metricName='mae')

bin_eval = BinaryClassificationEvaluator(labelCol='HIGH_COST', metricName='areaUnderROC')
mc_eval = MulticlassClassificationEvaluator(labelCol='HIGH_COST')

print("Full Metrics Table:")
print(f"Linear Regression: RMSE={reg_eval_rmse.evaluate(lr_pred):.2f}, R²={reg_eval_r2.evaluate(lr_pred):.4f}")
print(f"Random Forest: RMSE={reg_eval_rmse.evaluate(rf_pred):.2f}, R²={reg_eval_r2.evaluate(rf_pred):.4f}")
print(f"Logistic Regression: AUC={bin_eval.evaluate(log_pred):.4f}, Acc={mc_eval.evaluate(log_pred, {mc_eval.metricName:'accuracy'}):.4f}")
print(f"Decision Tree: AUC={bin_eval.evaluate(dt_pred):.4f}, Acc={mc_eval.evaluate(dt_pred, {mc_eval.metricName:'accuracy'}):.4f}")

Full Metrics Table:
Linear Regression: RMSE=10.65, R²=0.9978
Random Forest: RMSE=110.39, R²=0.7687
Logistic Regression: AUC=0.9981, Acc=0.9711
Decision Tree: AUC=0.9979, Acc=0.9924
